# 01 — Data Preparation

**Reduction or Relocation? NYC Congestion Pricing & Manhattan CBD Yellow-Taxi Ridership**

This notebook covers the **Data Understanding** and **Data Preparation** phases of CRISP-DM.
It ingests the twelve NYC TLC Yellow Taxi Trip Record Parquet files (Jan–Jun 2024 and
Jan–Jun 2025), filters to the Manhattan Central Business District (CBD), removes invalid
records while logging every exclusion, and aggregates valid trips into daily in-zone pickup
counts.

**Outputs**
- `../data/processed/daily_cbd_counts.csv` — one row per calendar day per period
- `../data/processed/cleaning_log.csv` — excluded-record counts by type and file


In [1]:
import duckdb
import pandas as pd
from pathlib import Path

pd.set_option("display.max_columns", 30)

RAW_DIR = Path("../data/raw")
LOOKUP_CSV = Path("../data/lookup/taxi_zone_lookup.csv")
OUT_DIR = Path("../data/processed")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# The twelve study files: Jan–Jun of 2024 (pre) and 2025 (post).
MONTHS = [f"{m:02d}" for m in range(1, 7)]
FILES = [(2024, m) for m in MONTHS] + [(2025, m) for m in MONTHS]

# --- Manhattan Central Business District taxi zones -------------------------------------
# The MTA Congestion Relief Zone is Manhattan AT OR BELOW 60th Street (excluding FDR Drive,
# the West Side Highway, and the Hugh L. Carey Tunnel surface connections). The TLC Taxi
# Zone Lookup CSV has no "below-60th" flag, so the 38 in-zone PULocationIDs are enumerated
# explicitly below. Manhattan has only 69 taxi zones in total; the remaining 31 lie north of
# 60th Street or are harbor/river islands, which is why the Task 2 proposal's "63 zones"
# figure is not supportable under a below-60th-Street definition.
CBD_ZONES = [
    4, 12, 13, 45, 48, 50, 68, 79, 87, 88, 90, 100, 107, 113, 114, 125, 137,
    144, 148, 158, 161, 162, 163, 164, 170, 186, 209, 211, 224, 229, 230, 231,
    232, 233, 234, 246, 249, 261,
]
print(f"CBD zones defined: {len(CBD_ZONES)}")
ZONE_SQL = ", ".join(str(z) for z in CBD_ZONES)

con = duckdb.connect()


CBD zones defined: 38


## Data Understanding

Confirm the schema, the presence of the required fields (`tpep_pickup_datetime`,
`PULocationID`, `trip_distance`, `fare_amount`), and the documented structural difference
between years: the 2025 files add a `cbd_congestion_fee` column. We select columns
explicitly during ingestion so this difference is handled transparently.

In [2]:
def schema_of(path):
    return con.execute(f"DESCRIBE SELECT * FROM read_parquet('{path}')").df()

s2024 = schema_of(RAW_DIR / "yellow_tripdata_2024-01.parquet")
s2025 = schema_of(RAW_DIR / "yellow_tripdata_2025-01.parquet")

cols_2024 = set(s2024.column_name)
cols_2025 = set(s2025.column_name)
print("Columns only in 2025 files:", sorted(cols_2025 - cols_2024))
print("Columns only in 2024 files:", sorted(cols_2024 - cols_2025))

required = ["tpep_pickup_datetime", "PULocationID", "trip_distance", "fare_amount"]
print("\nRequired fields present in both years:",
      all(c in cols_2024 and c in cols_2025 for c in required))
s2025[s2025.column_name.isin(required + ["cbd_congestion_fee"])]


Columns only in 2025 files: ['cbd_congestion_fee']
Columns only in 2024 files: []

Required fields present in both years: True


,column_name,column_type,null,key,default,extra
1,tpep_pickup_datetime,TIMESTAMP,YES,None,None,None
4,trip_distance,DOUBLE,YES,None,None,None
7,PULocationID,INTEGER,YES,None,None,None
10,fare_amount,DOUBLE,YES,None,None,None
19,cbd_congestion_fee,DOUBLE,YES,None,None,None


## Step 1 — Verify ingestion

Open every Parquet file and confirm a non-zero record count. This satisfies the first
criterion for success: complete ingestion of all twelve files with no missing months.

In [3]:
ingest = []
for year, month in FILES:
    path = RAW_DIR / f"yellow_tripdata_{year}-{month}.parquet"
    n = con.execute(f"SELECT COUNT(*) FROM read_parquet('{path}')").fetchone()[0]
    ingest.append({"year": year, "month": month, "file": path.name, "rows": n})

ingest_df = pd.DataFrame(ingest)
print(f"Files ingested: {len(ingest_df)}/12   |   total raw rows: {ingest_df.rows.sum():,}")
assert len(ingest_df) == 12 and (ingest_df.rows > 0).all(), "Missing or empty file!"
ingest_df


Files ingested: 12/12   |   total raw rows: 44,415,477


,year,month,file,rows
0,2024,01,yellow_tripdata_2024-01.parquet,2964624
1,2024,02,yellow_tripdata_2024-02.parquet,3007526
2,2024,03,yellow_tripdata_2024-03.parquet,3582628
3,2024,04,yellow_tripdata_2024-04.parquet,3514289
4,2024,05,yellow_tripdata_2024-05.parquet,3723833
5,2024,06,yellow_tripdata_2024-06.parquet,3539193
6,2025,01,yellow_tripdata_2025-01.parquet,3475226
7,2025,02,yellow_tripdata_2025-02.parquet,3577543
8,2025,03,yellow_tripdata_2025-03.parquet,4145257
9,2025,04,yellow_tripdata_2025-04.parquet,3970553


## Step 2 — Filter to CBD, clean, and log exclusions

For each file we restrict to CBD pickup zones and then classify every in-zone record into
exactly one mutually-exclusive bucket so the counts reconcile (`valid` + all exclusions =
in-CBD rows):

1. **`null_pickup`** — missing pickup timestamp (cannot be assigned to a day)
2. **`out_of_window`** — pickup timestamp falls outside that file's calendar month (TLC
   files contain a few stray timestamps from other periods); this also enforces the matched
   Jan 1–Jun 30 study windows
3. **`zero_distance`** — `trip_distance <= 0` (meter errors / cancelled trips)
4. **`nonpositive_fare`** — `fare_amount <= 0` (data-entry errors)

Valid records pass all four checks.

In [4]:
def month_bounds(year, month):
    start = pd.Timestamp(year=year, month=int(month), day=1)
    end = start + pd.offsets.MonthBegin(1)
    return start.strftime("%Y-%m-%d %H:%M:%S"), end.strftime("%Y-%m-%d %H:%M:%S")

def clean_query(path, start, end):
    return f"""
    WITH cbd AS (
        SELECT tpep_pickup_datetime AS pu, trip_distance AS dist, fare_amount AS fare
        FROM read_parquet('{path}')
        WHERE PULocationID IN ({ZONE_SQL})
    )
    SELECT CASE
        WHEN pu IS NULL THEN 'null_pickup'
        WHEN pu < TIMESTAMP '{start}' OR pu >= TIMESTAMP '{end}' THEN 'out_of_window'
        WHEN dist IS NULL OR dist <= 0 THEN 'zero_distance'
        WHEN fare IS NULL OR fare <= 0 THEN 'nonpositive_fare'
        ELSE 'valid' END AS reason,
        COUNT(*) AS n
    FROM cbd GROUP BY 1
    """

REASONS = ["valid", "null_pickup", "out_of_window", "zero_distance", "nonpositive_fare"]
log_rows = []
for year, month in FILES:
    path = RAW_DIR / f"yellow_tripdata_{year}-{month}.parquet"
    start, end = month_bounds(year, month)
    counts = dict.fromkeys(REASONS, 0)
    for reason, n in con.execute(clean_query(path, start, end)).fetchall():
        counts[reason] = n
    in_cbd = sum(counts.values())
    log_rows.append({
        "year": year, "month": month, "file": path.name,
        "in_cbd_rows": in_cbd, **{f"excl_{r}": counts[r] for r in REASONS if r != "valid"},
        "valid_trips": counts["valid"],
    })

cleaning_log = pd.DataFrame(log_rows)
cleaning_log


,year,month,file,in_cbd_rows,excl_null_pickup,excl_out_of_window,excl_zero_distance,excl_nonpositive_fare,valid_trips
0,2024,01,yellow_tripdata_2024-01.parquet,1696963,0,13,29449,19542,1647959
1,2024,02,yellow_tripdata_2024-02.parquet,1778496,0,10,36968,22111,1719407
2,2024,03,yellow_tripdata_2024-03.parquet,2129809,0,16,46702,32906,2050185
3,2024,04,yellow_tripdata_2024-04.parquet,2035634,0,7,20628,31057,1983942
4,2024,05,yellow_tripdata_2024-05.parquet,2132580,0,17,22087,33008,2077468
5,2024,06,yellow_tripdata_2024-06.parquet,2092648,0,36,25285,34394,2032933
6,2025,01,yellow_tripdata_2025-01.parquet,2020204,0,13,50504,75825,1893862
7,2025,02,yellow_tripdata_2025-02.parquet,2096057,0,26,54666,94802,1946563
8,2025,03,yellow_tripdata_2025-03.parquet,2406777,0,25,51952,105760,2249040
9,2025,04,yellow_tripdata_2025-04.parquet,2258764,0,2,45310,92509,2120943


### Cleaning summary

Report total in-zone records, the count by exclusion type, and the overall exclusion rate.
The proposal anticipated an exclusion rate under 1%.

In [5]:
excl_cols = ["excl_null_pickup", "excl_out_of_window", "excl_zero_distance", "excl_nonpositive_fare"]
totals = cleaning_log[["in_cbd_rows", *excl_cols, "valid_trips"]].sum()
total_excluded = totals[excl_cols].sum()
rate = total_excluded / totals["in_cbd_rows"] * 100

print(f"In-CBD records      : {totals['in_cbd_rows']:,}")
for c in excl_cols:
    print(f"  {c:<22}: {totals[c]:>10,}")
print(f"Total excluded      : {total_excluded:,}")
print(f"Valid trips         : {totals['valid_trips']:,}")
print(f"Exclusion rate      : {rate:.3f}%   (proposal expected < 1%)")
assert totals["valid_trips"] + total_excluded == totals["in_cbd_rows"], "Counts do not reconcile!"


In-CBD records      : 25,634,292
  excl_null_pickup      :          0
  excl_out_of_window    :        196
  excl_zero_distance    :    528,751
  excl_nonpositive_fare :    826,170
Total excluded      : 1,355,117
Valid trips         : 24,279,175
Exclusion rate      : 5.286%   (proposal expected < 1%)


## Step 3 — Aggregate valid trips to daily counts

Group the valid in-zone trips by calendar day and tag each row with its study period
(2024 = pre-intervention, 2025 = post-intervention).

In [6]:
def daily_query(path, start, end):
    return f"""
    SELECT CAST(tpep_pickup_datetime AS DATE) AS pickup_date, COUNT(*) AS trip_count
    FROM read_parquet('{path}')
    WHERE PULocationID IN ({ZONE_SQL})
      AND tpep_pickup_datetime IS NOT NULL
      AND tpep_pickup_datetime >= TIMESTAMP '{start}'
      AND tpep_pickup_datetime <  TIMESTAMP '{end}'
      AND trip_distance > 0
      AND fare_amount > 0
    GROUP BY 1 ORDER BY 1
    """

frames = []
for year, month in FILES:
    path = RAW_DIR / f"yellow_tripdata_{year}-{month}.parquet"
    start, end = month_bounds(year, month)
    frames.append(con.execute(daily_query(path, start, end)).df())

daily = pd.concat(frames, ignore_index=True)
daily["pickup_date"] = pd.to_datetime(daily["pickup_date"])
daily = daily.sort_values("pickup_date").reset_index(drop=True)
daily["period"] = daily["pickup_date"].dt.year
daily["day_of_year"] = daily["pickup_date"].dt.dayofyear

print("Days per period:")
print(daily.groupby("period").agg(days=("trip_count", "size"),
                                  mean_daily=("trip_count", "mean")).round(0))
daily.head()


Days per period:
        days  mean_daily
period                  
2024     182     63252.0
2025     181     70537.0


,pickup_date,trip_count,period,day_of_year
0,2024-01-01,45497,2024,1
1,2024-01-02,38943,2024,2
2,2024-01-03,43826,2024,3
3,2024-01-04,55019,2024,4
4,2024-01-05,53708,2024,5


## Step 4 — Validation

Confirm the analysis-ready dataset is sound:
- no nulls in `pickup_date` or `trip_count`
- every observed `PULocationID` resolves in the TLC Taxi Zone Lookup table
- each period spans roughly a full Jan–Jun window (~181–182 days)

In [7]:
# No nulls in the key columns
assert daily["pickup_date"].notna().all() and daily["trip_count"].notna().all()

# Every CBD zone resolves in the official lookup table (success criterion #3)
lookup = pd.read_csv(LOOKUP_CSV)
unresolved = set(CBD_ZONES) - set(lookup["LocationID"])
print("Unresolved CBD PULocationIDs:", unresolved or "none")
assert not unresolved, "Some CBD zones are missing from the lookup table!"

# Confirm all CBD zones are Manhattan and below-60th in spirit (Borough check)
cbd_lookup = lookup[lookup.LocationID.isin(CBD_ZONES)]
print("All CBD zones in Manhattan:", (cbd_lookup.Borough == "Manhattan").all())

# Day coverage
coverage = daily.groupby("period")["pickup_date"].agg(["min", "max", "size"])
print()
print(coverage)
assert daily.groupby("period").size().between(178, 183).all(), "Unexpected day coverage!"
print("\nValidation passed.")


Unresolved CBD PULocationIDs: none
All CBD zones in Manhattan: True

              min        max  size
period                            
2024   2024-01-01 2024-06-30   182
2025   2025-01-01 2025-06-30   181

Validation passed.


## Step 5 — Save outputs

Write the analysis-ready daily-count dataset and the data-cleaning log for use in
`02_data_analysis.ipynb`.

In [8]:
daily_out = daily[["pickup_date", "period", "day_of_year", "trip_count"]]
daily_out.to_csv(OUT_DIR / "daily_cbd_counts.csv", index=False)
cleaning_log.to_csv(OUT_DIR / "cleaning_log.csv", index=False)

print("Saved:")
print(f"  {OUT_DIR / 'daily_cbd_counts.csv'}  ({len(daily_out)} rows)")
print(f"  {OUT_DIR / 'cleaning_log.csv'}  ({len(cleaning_log)} rows)")
con.close()


Saved:
  ../data/processed/daily_cbd_counts.csv  (363 rows)
  ../data/processed/cleaning_log.csv  (12 rows)
